# 01 Core Workflows

Install Matheel, generate the tiny Java sample archive from source strings, then run preprocessing, chunking, lexical scoring, code metrics, archive ranking, and a comparison suite.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MATHEEL_REF = "v0.5.7"


def run(*args, cwd=None):
    subprocess.check_call(list(args), cwd=cwd)

run(sys.executable, "-m", "pip", "install", "jedi>=0.16,<1")
run("rm", "-rf", "/content/matheel")
run("git", "clone", "https://github.com/FahadEbrahim/matheel.git", "/content/matheel")
run("git", "checkout", MATHEEL_REF, cwd="/content/matheel")
run(sys.executable, "-m", "pip", "install", "-e", ".[all]", cwd="/content/matheel")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matheel_mplconfig")
print(f"Matheel installed from {MATHEEL_REF}")


In [ ]:
%cd /content/matheel
from pathlib import Path
from zipfile import ZipFile

from examples.sample_data import CODE_A_NAME, CODE_B_NAME, write_sample_archive

sample_archive = Path("/content/matheel/sample_pairs.zip")
write_sample_archive(sample_archive, overwrite=True)

with ZipFile(sample_archive) as archive:
    code_a = archive.read(CODE_A_NAME).decode("utf-8")
    code_b = archive.read(CODE_B_NAME).decode("utf-8")

print(f"Loaded {CODE_A_NAME} and {CODE_B_NAME}")

In [ ]:
from matheel.chunking import chunk_text
from matheel.preprocessing import preprocess_code

print("=== Basic preprocessing ===")
print(preprocess_code(code_a, mode="basic"))
print("
=== Code chunks ===")
for index, chunk in enumerate(chunk_text(code_a, method="code", chunk_size=64, max_chunks=2, chunk_language="java"), start=1):
    print(f"Chunk {index}:")
    print(chunk)
    print()

In [ ]:
from matheel.similarity import calculate_similarity

lexical_scores = {
    "levenshtein": calculate_similarity(code_a, code_b, feature_weights={"levenshtein": 1.0}),
    "jaro_winkler": calculate_similarity(code_a, code_b, feature_weights={"jaro_winkler": 1.0}),
    "winnowing": calculate_similarity(code_a, code_b, feature_weights={"winnowing": 1.0}, winnowing_kgram=5, winnowing_window=4),
    "gst": calculate_similarity(code_a, code_b, feature_weights={"gst": 1.0}, gst_min_match_length=5),
}
for name, score in lexical_scores.items():
    print(name, round(score, 4))

In [ ]:
semantic_blend = calculate_similarity(
    code_a,
    code_b,
    model_name="huggingface/CodeBERTa-small-v1",
    feature_weights={"semantic": 0.7, "levenshtein": 0.3},
)
print("semantic_blend", round(semantic_blend, 4))

In [ ]:
code_metrics = {
    "codebleu": calculate_similarity(code_a, code_b, code_metric="codebleu", code_language="java", code_metric_weight=1.0, feature_weights={"code_metric": 1.0}),
    "crystalbleu": calculate_similarity(code_a, code_b, code_metric="crystalbleu", code_language="java", code_metric_weight=1.0, crystalbleu_trivial_ngram_count=0, feature_weights={"code_metric": 1.0}),
    "ruby": calculate_similarity(code_a, code_b, code_metric="ruby", code_language="java", code_metric_weight=1.0, feature_weights={"code_metric": 1.0}),
}
for name, score in code_metrics.items():
    print(name, round(score, 4))

In [ ]:
from matheel.comparison_suite import run_comparison_suite
from matheel.similarity import get_sim_list

results = get_sim_list(sample_archive, threshold=0.2, number_results=10, feature_weights={"levenshtein": 1.0})
display(results.head().round(4))

runs = [
    {"run_name": "levenshtein", "options": {"feature_weights": {"levenshtein": 1.0}}},
    {"run_name": "lexical_blend", "options": {"feature_weights": {"levenshtein": 0.6, "jaro_winkler": 0.4}}},
]
summary, details = run_comparison_suite(sample_archive, runs, reproducibility_out="/content/matheel/core_reproducibility.json")
display(summary.round(4))
display(details["levenshtein"].head().round(4))